# m=2 Feature Geometry — Trajectories & Argmax Regions

For each n ∈ {16, 32, 64, 128} and l ∈ {1, 2, 3, 4} and S ∈ {0.85, 0.9, 0.95}, plot:

1. **Feature trajectories**: for each input feature i, trace the parametric curve `z(t·e_i)` for t ∈ [0, 1]. Shows where each feature lives in the 2D bottleneck and whether its trajectory is straight (positive homogeneous, no bias) or curved (ReLU kinks visible).

2. **Argmax-feature regions**: for a grid of (z1, z2) points in the bottleneck, decode → identify the dominant feature → color the region. Shows how the decoder partitions the bottleneck space.

This uses the FINAL post-frontier-push models — the cleanest landscape we have.


In [ ]:
import os, sys, math, json
sys.path.insert(0, '../code')
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from itertools import cycle

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

import core
core.device = device
from core import Autoencoder, generate_sparse_data
print('Device:', device)


In [ ]:
def load_model(n, m, l, S):
    path = Path(f'results_db/models/model_n{n}_m{m}_l{l}_S{S}.pt')
    if not path.exists():
        return None
    model = Autoencoder(n, m, l, tied_weights=(l==1)).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model


def feature_trajectories(model, n_steps=50, t_max=1.0):
    """For each feature i, compute z(t * e_i) for t in [0, t_max]."""
    n = model.n
    ts = torch.linspace(0, t_max, n_steps, device=device)
    trajectories = np.zeros((n, n_steps, 2))  # [feature, step, z_dim]
    with torch.no_grad():
        for i in range(n):
            e = torch.zeros(n, device=device)
            e[i] = 1.0
            x = torch.outer(ts, e)  # [n_steps, n]
            z = model.encode(x)  # [n_steps, 2]
            trajectories[i] = z.cpu().numpy()
    return trajectories


def argmax_region_grid(model, n_grid=200, z_max=None):
    """Color the (z1, z2) plane by argmax(decode(z))."""
    n = model.n
    # Auto-detect z range from data activations
    if z_max is None:
        with torch.no_grad():
            x = generate_sparse_data(2000, n, 0.9)
            z_data = model.encode(x).cpu().numpy()
            z_max = max(abs(z_data.min()), abs(z_data.max())) * 1.4
    z = np.linspace(-z_max, z_max, n_grid)
    zz1, zz2 = np.meshgrid(z, z)
    Z = np.stack([zz1.ravel(), zz2.ravel()], axis=1)  # [n_grid^2, 2]
    Zt = torch.tensor(Z, dtype=torch.float32, device=device)
    with torch.no_grad():
        x_recon = model.decode(Zt)  # [n_grid^2, n]
        argmax = x_recon.argmax(dim=1).cpu().numpy()
        max_val = x_recon.max(dim=1).values.cpu().numpy()
    argmax_grid = argmax.reshape(n_grid, n_grid)
    max_val_grid = max_val.reshape(n_grid, n_grid)
    return zz1, zz2, argmax_grid, max_val_grid, z_max


In [ ]:
import colorsys

def sci_palette(n, sat=0.7, val=0.88, scramble=37):
    """Same palette as the headline image: HSV-scrambled, moderate sat/val."""
    arr = []
    for i in range(n):
        hue = ((i * scramble) % n) / n
        v = val + 0.04 * ((i // 8) % 3 - 1)
        s = sat + 0.05 * ((i // 5) % 3 - 1)
        arr.append(colorsys.hsv_to_rgb(hue, s, v))
    return arr


def plot_m2_panel(model, ax_traj, ax_region, n_grid=300, max_features_traj=24):
    """Plot trajectories (a subset for large n) and argmax regions side by side.
    Uses the same scientific palette + log-magnitude contours as the headline image."""
    n = model.n
    palette = sci_palette(n)
    cmap = mcolors.ListedColormap(palette)

    # Trajectories — subsample features if n is large
    trajectories = feature_trajectories(model, n_steps=40)
    if n <= max_features_traj:
        feature_idxs = list(range(n))
    else:
        feature_idxs = np.linspace(0, n-1, max_features_traj).astype(int).tolist()

    for i in feature_idxs:
        traj = trajectories[i]
        color = palette[i]
        ax_traj.plot(traj[:, 0], traj[:, 1], color=color, alpha=0.85, lw=1.3)
        ax_traj.plot(traj[-1, 0], traj[-1, 1], 'o', color=color, markersize=4,
                     markeredgecolor='black', markeredgewidth=0.4)
    ax_traj.axhline(0, color='gray', lw=0.5, alpha=0.5)
    ax_traj.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax_traj.set_aspect('equal', adjustable='box')
    n_shown = len(feature_idxs)
    ax_traj.set_title(f'traj n={n} m=2 l={model.l}' +
                      (f' ({n_shown}/{n})' if n_shown < n else ''),
                      fontsize=10)

    # Argmax regions + log-magnitude contours
    zz1, zz2, argmax, maxval, z_max = argmax_region_grid(model, n_grid=n_grid)
    ax_region.imshow(argmax, extent=[-z_max, z_max, -z_max, z_max],
                     origin='lower', cmap=cmap, vmin=0, vmax=n - 1,
                     interpolation='nearest')

    # Log-magnitude contours overlay (same as headline image)
    xs = np.linspace(-z_max, z_max, n_grid)
    XX, YY = np.meshgrid(xs, xs, indexing='xy')
    Z = np.stack([XX, YY], axis=-1).reshape(-1, 2)
    with torch.no_grad():
        x_hat = model.decode(torch.tensor(Z, dtype=torch.float32, device=device)).cpu().numpy()
    dec_mag = np.log1p(np.linalg.norm(x_hat, axis=1)).reshape(n_grid, n_grid)
    levels = np.linspace(dec_mag.min(), dec_mag.max(), 14)[2:-1]
    ax_region.contour(XX, YY, dec_mag, levels=levels, colors='#222',
                      linewidths=0.45, alpha=0.65)

    ax_region.set_aspect('equal', adjustable='box')
    ax_region.set_title(f'regions n={n} m=2 l={model.l}', fontsize=10)


## Grid of trajectories + regions for all (n, l) at fixed S=0.9

In [ ]:
ns = [16, 32, 64, 128]
ls = [1, 2, 3, 4]
S = 0.9

fig, axes = plt.subplots(len(ns), len(ls) * 2, figsize=(4.0*len(ls), 4.5*len(ns)))
for i, n in enumerate(ns):
    for j, l in enumerate(ls):
        model = load_model(n, 2, l, S)
        ax_traj = axes[i, 2*j]
        ax_region = axes[i, 2*j+1]
        if model is None:
            ax_traj.text(0.5, 0.5, 'no model', ha='center', va='center', transform=ax_traj.transAxes)
            ax_region.set_visible(False)
            continue
        plot_m2_panel(model, ax_traj, ax_region, n_grid=160)

plt.suptitle(f'm=2 Feature Geometry  (S={S})', fontsize=14)
plt.tight_layout()
plt.savefig('fig_m2_geometry_S0.9.png', dpi=80, bbox_inches='tight')
plt.show()
print('saved fig_m2_geometry_S0.9.png')


## Same for S=0.95 (more sparse)

In [ ]:
S = 0.95
fig, axes = plt.subplots(len(ns), len(ls) * 2, figsize=(4.0*len(ls), 4.5*len(ns)))
for i, n in enumerate(ns):
    for j, l in enumerate(ls):
        model = load_model(n, 2, l, S)
        ax_traj = axes[i, 2*j]
        ax_region = axes[i, 2*j+1]
        if model is None:
            ax_traj.text(0.5, 0.5, 'no model', ha='center', va='center', transform=ax_traj.transAxes)
            ax_region.set_visible(False)
            continue
        plot_m2_panel(model, ax_traj, ax_region, n_grid=160)

plt.suptitle(f'm=2 Feature Geometry  (S={S})', fontsize=14)
plt.tight_layout()
plt.savefig('fig_m2_geometry_S0.95.png', dpi=80, bbox_inches='tight')
plt.show()
print('saved fig_m2_geometry_S0.95.png')


## And S=0.85 (denser)

In [ ]:
S = 0.85
fig, axes = plt.subplots(len(ns), len(ls) * 2, figsize=(4.0*len(ls), 4.5*len(ns)))
for i, n in enumerate(ns):
    for j, l in enumerate(ls):
        model = load_model(n, 2, l, S)
        ax_traj = axes[i, 2*j]
        ax_region = axes[i, 2*j+1]
        if model is None:
            ax_traj.text(0.5, 0.5, 'no model', ha='center', va='center', transform=ax_traj.transAxes)
            ax_region.set_visible(False)
            continue
        plot_m2_panel(model, ax_traj, ax_region, n_grid=160)

plt.suptitle(f'm=2 Feature Geometry  (S={S})', fontsize=14)
plt.tight_layout()
plt.savefig('fig_m2_geometry_S0.85.png', dpi=80, bbox_inches='tight')
plt.show()
print('saved fig_m2_geometry_S0.85.png')
